In [4]:
import numpy as np

from utils.dgp import DataGenerator, heiss_x_sampler, heiss_beta_support_probs
from utils.estimators import FKRBEstimator
from utils.visualization import plot_cdf_3D

In [5]:
J = 4
K = 2
N_SET = (1000, 10000)
R_SET_DISCRETE = (25, 81, 289)
S_SET_DISCRETE = (17, 49, 161)

OUTER_REPETITIONS = 200
INNER_REPETITIONS = 50
REPETITIONS = 5

SEED = 42

rng = np.random.default_rng(SEED)


In [8]:
def get_mean(support, probs):
    return support @ probs

for N in N_SET:
    for R in R_SET_DISCRETE:
        coverage_rates = np.zeros(REPETITIONS)
        coverage_rates_1 = np.zeros(REPETITIONS)
        coverage_rates_2 = np.zeros(REPETITIONS)
        
        full_grid, support, probs = heiss_beta_support_probs(R)

        support_1 = support[:,0]
        support_2 = support[:,1]

        true_mean_1 = get_mean(support_1,probs)
        true_mean_2 = get_mean(support_2,probs)
        
        generator = DataGenerator(N=N, J=J, K=K, rng=rng)
        
        for i in range(REPETITIONS):
            generator.reset()
            y, x = generator.generate(x_sampler=heiss_x_sampler, beta_support=support, beta_probs=probs)

            estimator = FKRBEstimator(beta_support=full_grid)
            theta = estimator.estimate(y=y, x=x)

            ci = estimator.get_confidence_interval(alpha=0.5)
            _, ci_1 = estimator.plug_in_estimate_functional(lambda probs: get_mean(support_1,probs), ci = True,alpha=0.5)
            _, ci_2 = estimator.plug_in_estimate_functional(lambda probs: get_mean(support_2,probs), ci = True,alpha=0.5)

            in_interval = (probs >= ci[:, 0]) & (probs <= ci[:, 1])
            in_interval_1 = (true_mean_1 >= ci_1[0]) & (true_mean_1 <= ci_1[1])
            in_interval_2 = (true_mean_2 >= ci_2[0]) & (true_mean_2 <= ci_2[1])

            coverage_rates[i] = np.mean(in_interval)
            coverage_rates_1[i] = in_interval_1
            coverage_rates_2[i] = in_interval_2
        
        print('regular')
        print(np.mean(coverage_rates))
        print('first')
        print(np.mean(coverage_rates_1))
        print('second')
        print(np.mean(coverage_rates_2))

            # support_1 = support[:,0]
            # support_2 = support[:,1]

            # estimated_mean_1, ci_1 = estimator.plug_in_estimate_functional(lambda probs: get_mean(support_1,probs), ci = True)
            # estimated_mean_2, ci_2 = estimator.plug_in_estimate_functional(lambda probs: get_mean(support_2,probs), ci = True)
            
            # coverage_rates = np.zeros(INNER_REPETITIONS)
            # coverage_rates_1 = np.zeros(INNER_REPETITIONS)
            # coverage_rates_2 = np.zeros(INNER_REPETITIONS)
            # for i in range(INNER_REPETITIONS):
            #     generator.reset()
            #     y, x = generator.generate(x_sampler=heiss_x_sampler, beta_support=support, beta_probs=probs)

            #     estimator = FKRBEstimator(beta_support=full_grid)
            #     theta = estimator.estimate(y=y, x=x, estimate_both=False)
            #     estimated_mean_1 = estimator.plug_in_estimate_functional(lambda probs: get_mean(support_1,probs), ci = False)
            #     estimated_mean_2 = estimator.plug_in_estimate_functional(lambda probs: get_mean(support_2,probs), ci = False)

            #     in_interval = (theta >= ci[:, 0]) & (theta <= ci[:, 1])
            #     in_interval_1 = (estimated_mean_1 >= ci_1[0]) & (estimated_mean_1 <= ci_1[1])
            #     in_interval_2 = (estimated_mean_2 >= ci_2[0]) & (estimated_mean_2 <= ci_2[1])

            #     coverage_rates[i] = np.mean(in_interval)
            #     coverage_rates_1[i] = in_interval_1
            #     coverage_rates_2[i] = in_interval_2
            
            # print("regular")
            # print(np.mean(coverage_rates))
            # print("first variable")
            # print(np.mean(coverage_rates_1))
            # print("second variable")
            # print(np.mean(coverage_rates_2))
            

regular
0.9199999999999999
first
1.0
second
1.0
regular
1.0
first
1.0
second
1.0
regular
1.0
first
1.0
second
1.0
regular
0.744
first
1.0
second
1.0
regular
1.0
first
1.0
second
1.0
regular
1.0
first
1.0
second
1.0
